# E0006 — GPU runtime preflight (L4 x4, NO MODEL LOAD)

Purpose: validate the frozen offline CUDA 12.9 runtime on the actual Kaggle L4 x4 environment **without loading Nemotron weights**.

Required settings:
- Accelerator: **GPU L4 x4**
- Internet: **OFF**
- Attach the ARC Prize 2026 competition if Kaggle requires it to expose L4 x4
- Attach the saved wheelhouse-builder notebook output containing `e0006_cu129_wheelhouse/`
- Do **not** attach/load the Nemotron model for this preflight

This gate installs the runtime in `/tmp`, checks four L4 devices, runs one tiny CUDA op per GPU, imports the Nemotron vLLM class, and runs a 4-process NCCL all-reduce. It does not allocate model weights.

Expected output: `/kaggle/working/e0006_gpu_runtime_preflight.json`


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

OUT = Path("/kaggle/working/e0006_gpu_runtime_preflight.json")
INPUT_ROOT = Path("/kaggle/input")
RUNTIME = Path("/tmp/e0006_runtime")
EXPECTED_VLLM = "0.27.1+cu129"
EXPECTED_TORCH = "2.13.0+cu129"
EXPECTED_FLASHINFER = "0.6.16.post3"

def find_one(pattern: str) -> Path:
    hits = sorted(INPUT_ROOT.rglob(pattern))
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one {pattern} under /kaggle/input; found {len(hits)}")
    return hits[0]

def gpu_snapshot():
    exe = shutil.which("nvidia-smi")
    if not exe:
        return []
    cmd = [exe, "--query-gpu=index,name,memory.total,memory.free,driver_version",
           "--format=csv,noheader,nounits"]
    cp = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
    rows = []
    if cp.returncode == 0:
        for line in cp.stdout.splitlines():
            f = [x.strip() for x in line.split(",")]
            if len(f) == 5:
                idx, name, total, free, driver = f
                rows.append({
                    "index": int(idx), "name": name,
                    "memory_total_mib": int(total),
                    "memory_free_mib": int(free),
                    "driver_version": driver,
                })
    return rows

payload = {
    "experiment": "E0006",
    "gate": "D4_GPU_RUNTIME_PREFLIGHT_L4X4",
    "status": "STARTING",
    "internet_required": False,
    "gpu_snapshot_before": gpu_snapshot(),
    "python": sys.version,
}

try:
    vllm_wheel = find_one("vllm-0.27.1+cu129-*.whl")
    cubin_wheel = find_one("flashinfer_cubin-0.6.16.post3-*.whl")
    wheelhouse = vllm_wheel.parent
    payload["wheelhouse"] = str(wheelhouse)
    payload["wheelhouse_file_count"] = len(list(wheelhouse.glob("*.whl")))

    if RUNTIME.exists():
        shutil.rmtree(RUNTIME)
    RUNTIME.mkdir(parents=True)

    install_cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", str(wheelhouse),
        "--target", str(RUNTIME),
        "--ignore-installed", "--no-cache-dir", "--no-compile",
        str(vllm_wheel),
        "flashinfer-cubin==0.6.16.post3",
    ]
    payload["install_command"] = install_cmd
    t0 = time.time()
    cp = subprocess.run(install_cmd, capture_output=True, text=True, timeout=2400)
    payload["install_seconds"] = round(time.time() - t0, 3)
    payload["install_returncode"] = cp.returncode
    payload["install_stdout_tail"] = cp.stdout[-8000:]
    payload["install_stderr_tail"] = cp.stderr[-8000:]
    if cp.returncode != 0:
        raise RuntimeError("offline isolated runtime install failed")

    env = os.environ.copy()
    env["PYTHONPATH"] = str(RUNTIME) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    env["PATH"] = str(RUNTIME / "bin") + os.pathsep + env.get("PATH", "")
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env["VLLM_NO_USAGE_STATS"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"

    probe_code = r"""
import importlib.metadata as md
import json
import torch
import flashinfer
import vllm
import triton
from vllm.model_executor.models.nemotron_h import NemotronHForCausalLM

r = {
    "versions": {
        "torch": md.version("torch"),
        "vllm": md.version("vllm"),
        "triton": md.version("triton"),
        "flashinfer-python": md.version("flashinfer-python"),
        "flashinfer-cubin": md.version("flashinfer-cubin"),
    },
    "torch_cuda_build": torch.version.cuda,
    "torch_cuda_available": bool(torch.cuda.is_available()),
    "torch_cuda_device_count": int(torch.cuda.device_count()),
    "devices": [],
    "nemotron_class_import": NemotronHForCausalLM.__name__,
}
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    x = torch.tensor([float(i + 1)], device=f"cuda:{i}")
    y = (x * 2).item()
    r["devices"].append({
        "index": i,
        "name": p.name,
        "total_memory_gib": round(p.total_memory / 1024**3, 3),
        "capability": [int(p.major), int(p.minor)],
        "simple_cuda_value": y,
    })
print(json.dumps(r, sort_keys=True))
"""
    t1 = time.time()
    probe = subprocess.run(
        [sys.executable, "-c", probe_code],
        capture_output=True, text=True, env=env, timeout=300,
    )
    payload["gpu_import_probe_seconds"] = round(time.time() - t1, 3)
    payload["gpu_import_probe_returncode"] = probe.returncode
    payload["gpu_import_probe_stdout"] = probe.stdout[-12000:]
    payload["gpu_import_probe_stderr"] = probe.stderr[-12000:]
    if probe.returncode != 0:
        raise RuntimeError("GPU import/CUDA probe failed")

    probe_payload = json.loads(probe.stdout.strip().splitlines()[-1])
    payload["gpu_import_probe"] = probe_payload

    nccl_script = Path("/tmp/e0006_nccl_probe.py")
    nccl_script.write_text(r"""
import json, os
import torch
import torch.distributed as dist

dist.init_process_group("nccl")
rank = dist.get_rank()
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
x = torch.tensor([float(rank + 1)], device=f"cuda:{local_rank}")
dist.all_reduce(x, op=dist.ReduceOp.SUM)
result = {"rank": rank, "local_rank": local_rank, "sum": float(x.item())}
print("E0006_NCCL " + json.dumps(result, sort_keys=True), flush=True)
dist.destroy_process_group()
""", encoding="utf-8")

    nccl_cmd = [
        sys.executable, "-m", "torch.distributed.run",
        "--standalone", "--nproc-per-node=4", str(nccl_script)
    ]
    t2 = time.time()
    nccl = subprocess.run(nccl_cmd, capture_output=True, text=True, env=env, timeout=180)
    payload["nccl_probe_seconds"] = round(time.time() - t2, 3)
    payload["nccl_probe_returncode"] = nccl.returncode
    payload["nccl_probe_stdout"] = nccl.stdout[-12000:]
    payload["nccl_probe_stderr"] = nccl.stderr[-12000:]

    nccl_rows = []
    for line in nccl.stdout.splitlines():
        if line.startswith("E0006_NCCL "):
            nccl_rows.append(json.loads(line.split(" ", 1)[1]))
    payload["nccl_results"] = nccl_rows

    versions = probe_payload["versions"]
    devices = probe_payload["devices"]
    runtime_ok = (
        versions.get("torch") == EXPECTED_TORCH
        and versions.get("vllm") == EXPECTED_VLLM
        and versions.get("flashinfer-python") == EXPECTED_FLASHINFER
        and versions.get("flashinfer-cubin") == EXPECTED_FLASHINFER
        and probe_payload.get("torch_cuda_build") == "12.9"
        and probe_payload.get("torch_cuda_available") is True
        and probe_payload.get("torch_cuda_device_count") == 4
        and len(devices) == 4
        and all(d.get("name") == "NVIDIA L4" for d in devices)
        and all(d.get("capability") == [8, 9] for d in devices)
        and nccl.returncode == 0
        and len(nccl_rows) == 4
        and all(abs(float(r.get("sum", -1)) - 10.0) < 1e-6 for r in nccl_rows)
    )
    payload["runtime_ok"] = runtime_ok
    payload["status"] = "GPU_RUNTIME_READY" if runtime_ok else "BLOCKED_GPU_RUNTIME"
    payload["gpu_snapshot_after"] = gpu_snapshot()
except Exception as e:
    payload["status"] = "BLOCKED_GPU_RUNTIME"
    payload["error"] = f"{type(e).__name__}: {e}"

OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "status": payload["status"],
    "install_seconds": payload.get("install_seconds"),
    "gpu_import_probe": payload.get("gpu_import_probe"),
    "nccl_results": payload.get("nccl_results"),
    "error": payload.get("error"),
}, indent=2, sort_keys=True))
print(f"\nWROTE: {OUT}")
